### Imports:


### Imports

In [2]:
import tensorflow as tf
#prevent TF from grabbing all GPU memory at once
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [3]:
from tensorflow.keras import mixed_precision

# Tell Keras to use float16 for memory, but keep float32 for numeric stability in the loss
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

print('Compute dtype: %s' % policy.compute_dtype)
print('Variable dtype: %s' % policy.variable_dtype)

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3060 Laptop GPU, compute capability 8.6
Compute dtype: float16
Variable dtype: float32


In [4]:
import os
import sys
import keras.applications
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
from keras.applications.efficientnet import preprocess_input


if os.getcwd().endswith('models'):
    os.chdir('..')
    
from utils.utils_model import *
from utils.utils_augmentation import *
from utils.utils_preproc import *

In [5]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
print("TF built with CUDA:", tf.test.is_built_with_cuda())
print("GPU available to TF:", tf.test.is_gpu_available())  # deprecated but still works

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TF built with CUDA: True
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
GPU available to TF: True


In [1]:
# COLAB

from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/MDSAA-DS/DL-project/dataverse_files/data"

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np
import json
from PIL import Image
import matplotlib.pyplot as plt

if os.getcwd().endswith('models'):
    os.chdir('..')

# from utils.utils_model import *

with open(os.path.join(base_path, "label2idx.json"), "r") as f:
    label2idx = json.load(f)

### utils

In [6]:
# Load label mapping first — everything else depends on it
with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

N_CLASSES  = len(label2idx)
BATCH_SIZE = 32

train_df = pd.read_csv('data/augmented_metadata.csv')
val_df   = pd.read_csv('data/val_split.csv')
test_df  = pd.read_csv('data/test_split.csv')

# Normalise column names and encode labels for all splits
for df in [train_df, val_df, test_df]:
    if 'cleaned_path' in df.columns and 'image_path' in df.columns:
        df.drop(columns=['image_path'], inplace=True)
    df.rename(columns={'cleaned_path': 'image_path'}, inplace=True)
    df['dx_encoded'] = df['dx'].map(label2idx).astype(int)

# Build datasets
train_ds = make_dataset(train_df, shuffle=True, repeat=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE
class_weights_dict = make_class_weights(train_df)

In [7]:
print(f"Steps per epoch: {STEPS_PER_EPOCH}")
print(f"Class weights: {class_weights_dict}")

Steps per epoch: 394
Class weights: {0: 1.4493569131832797, 1: 1.2340862422997947, 2: 0.8370473537604457, 3: 2.504166666666667, 4: 0.8727008712487899, 5: 0.42867332382310985, 6: 2.3415584415584414}


In [3]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import keras
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
#from utils.utils_preproc import *


def format_center_crop_tf(img):
    shape     = tf.shape(img)
    h, w      = shape[0], shape[1]
    crop_size = tf.minimum(h, w)
    offset_h  = (h - crop_size) // 2
    offset_w  = (w - crop_size) // 2
    img = tf.image.crop_to_bounding_box(img, offset_h, offset_w, crop_size, crop_size)
    img = tf.image.resize(img, [224, 224])
    return img

# ── Dataset ─────────────────────────────────────────────────────────────────
def load_image_new(path, label, resize_function=format_center_crop_tf, preprocess_fn=None):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32)
    img = resize_function(img)
    if preprocess_fn is not None:
        img = preprocess_fn(img)
    return img, label

def make_dataset_new(df, shuffle=False, repeat=False, batch_size=32):
    paths  = df["image_path"].values.reshape(-1)
    labels = df["dx_encoded"].values.reshape(-1)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(
        lambda x, y: load_image_new(x, y),
        num_parallel_calls=2
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=min(1000, len(df)))
    if repeat:
        ds = ds.repeat()
    ds = ds.batch(batch_size).prefetch(1)
    return ds

# ── Class Weights ─────────────────────────────────────────────────────────────────
def make_class_weights(df):
    classes = np.array(sorted(df["dx_encoded"].unique()))
    class_weights = compute_class_weight(class_weight="balanced",classes=classes,y=df["dx_encoded"])
    return {i: w for i, w in enumerate(class_weights)}


# ── Callbacks ─────────────────────────────────────────────────────────────────

def get_callbacks(checkpoint_path, patience_es=8, patience_lr=4, monitor="val_loss", mode="min"):
    """
    Returns the standard callback list used by all models.

    Parameters
    ----------
    checkpoint_path : str
        Path where the best weights will be saved, e.g.
        "checkpoints/model_a_best.weights.h5"
    patience_es : int
        Number of epochs with no improvement before EarlyStopping
        halts training and restores the best weights.
    patience_lr : int
        Number of epochs with no improvement before ReduceLROnPlateau
        halves the learning rate.
    monitor : str
        Metric to be monitored (e.g., "val_loss", "val_accuracy").
    mode : str
        One of {"auto", "min", "max"}. In "min" mode, training will
        stop when the quantity monitored has stopped decreasing.

    Returns
    -------
    list[keras.callbacks.Callback]
    """

    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=monitor,
            mode=mode,
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor=monitor,
            mode=mode,
            patience=patience_es,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=monitor,
            mode=mode,
            factor=0.5,
            patience=patience_lr,
            min_lr=1e-7,
            verbose=1,
        ),
    ]


class BatchTimeCallback(tf.keras.callbacks.Callback):
    """
    Custom Keras callback that records the duration of every
    training batch and epoch.

    After training, the recorded times can be used to compare computational
    cost across different augmentation strategies (offline vs. online).

    Attributes:
        batch_times (list[float]): Duration of each training batch in seconds.
        epoch_times (list[float]): Duration of each epoch in seconds.
    """

    def on_train_begin(self, logs=None):
        """Initialize empty lists to store batch and epoch durations."""
        self.batch_times = []
        self.epoch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        """Record the start timestamp of the current epoch."""
        self._epoch_start = time.time()

    def on_train_batch_begin(self, batch, logs=None):
        """Record the start timestamp of the current batch."""
        self._batch_start = time.time()

    def on_train_batch_end(self, batch, logs=None):
        """Compute and store the elapsed time for the completed batch."""
        self.batch_times.append(time.time() - self._batch_start)

    def on_epoch_end(self, epoch, logs=None):
        """Compute and store the elapsed time for the completed epoch."""
        self.epoch_times.append(time.time() - self._epoch_start)


# ── Plotting ──────────────────────────────────────────────────────────────────

def plot_history(history, title):
    """
    Plots loss and accuracy curves for a single Keras training History object.

    Parameters
    ----------
    history : keras.callbacks.History
        Object returned by model.fit().
    title : str
        Figure title, e.g. "Model B — EfficientNetB0 Phase 2".
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title)

    axes[0].plot(history.history["loss"],         label="train")
    axes[0].plot(history.history["val_loss"],     label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"],     label="train")
    axes[1].plot(history.history["val_accuracy"], label="val")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

# ── Metrics ────────────────────────────────────────────────────────────────── #need change

class BalancedAccuracy(tf.keras.metrics.Metric):
    """Balanced accuracy = mean per-class recall. Works with sparse labels + logits."""
    def __init__(self, num_classes, name='balanced_accuracy', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.tp = self.add_weight(name='tp', shape=(num_classes,), initializer='zeros')
        self.totals = self.add_weight(name='totals', shape=(num_classes,), initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(tf.squeeze(y_true), tf.int32)
        y_pred = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
        correct = tf.cast(tf.equal(y_true, y_pred), tf.float32)
        ones = tf.ones_like(y_true, dtype=tf.float32)
        self.totals.assign_add(tf.math.unsorted_segment_sum(ones, y_true, self.num_classes))
        self.tp.assign_add(tf.math.unsorted_segment_sum(correct, y_true, self.num_classes))

    def result(self):
        return tf.reduce_mean(tf.math.divide_no_nan(self.tp, self.totals))

    def reset_state(self):
        self.tp.assign(tf.zeros(self.num_classes))
        self.totals.assign(tf.zeros(self.num_classes))




# ── Evaluation ────────────────────────────────────────────────────────────────

def evaluate_model(model, test_ds, label2idx, model_name="model"):
    """
    Full evaluation suite for a trained Keras model.

    Outputs
    -------
    - Classification report (precision, recall, F1 per class)
    - Melanoma recall highlighted separately (clinical priority metric)
    - Confusion matrix heatmap
    - Macro AUC-ROC score
    - Per-class ROC curves (melanoma plotted with a thicker line)

    Parameters
    ----------
    model : keras.Model
        Trained model. Assumed to output raw logits (from_logits=True).
    test_ds : tf.data.Dataset
        Unbatched or batched test dataset yielding (images, labels).
    label2idx : dict
        Mapping from class name to integer index, e.g. {"mel": 0, "nv": 1, ...}
    model_name : str
        Label used in plot titles and printed headers.

    Returns
    -------
    dict with keys "macro_auc" and "mel_recall"
    """
    n_classes  = len(label2idx)
    idx2label  = {v: k for k, v in label2idx.items()}
    class_names = [idx2label[i] for i in range(n_classes)]

    # ── Collect predictions ───────────────────────────────────────────────────
    y_true, y_pred_proba = [], []

    for images, labels in test_ds:
        proba = tf.nn.softmax(model(images, training=False)).numpy()
        y_pred_proba.append(proba)
        y_true.extend(labels.numpy())

    y_pred_proba = np.vstack(y_pred_proba)
    y_true       = np.array(y_true)
    y_pred       = np.argmax(y_pred_proba, axis=1)

    # ── Classification report ─────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  {model_name} — classification report")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, target_names=class_names))

    mel_idx    = label2idx["mel"]
    mel_recall = (y_pred[y_true == mel_idx] == mel_idx).mean()
    print(f"  *** Melanoma recall: {mel_recall:.3f} ***  (clinical priority metric)")

    # ── Confusion matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
    )
    plt.title(f"{model_name} — confusion matrix")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

    # ── Macro AUC-ROC ─────────────────────────────────────────────────────────
    y_bin     = label_binarize(y_true, classes=list(range(n_classes)))
    macro_auc = roc_auc_score(
        y_bin, y_pred_proba, average="macro", multi_class="ovr"
    )
    print(f"\n  Macro AUC-ROC: {macro_auc:.4f}")

    # ── Per-class ROC curves ──────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 7))
    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_proba[:, i])
        auc_score   = roc_auc_score(y_bin[:, i], y_pred_proba[:, i])
        lw = 2.5 if cls == "mel" else 1.2
        ax.plot(fpr, tpr, lw=lw, label=f"{cls} (AUC={auc_score:.3f})")

    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"{model_name} — per-class ROC curves  (melanoma in bold)")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    return {"macro_auc": macro_auc, "mel_recall": mel_recall}


# ── Final comparison plot ──────────────────────────────────────────────────────

def plot_comparison(results: dict):
    """
    Bar chart comparing all trained models side by side.

    Parameters
    ----------
    results : dict
        Keys are model names, values are dicts with "macro_auc" and
        "mel_recall". Example:
            {
                "A — Custom CNN":      {"macro_auc": 0.91, "mel_recall": 0.78},
                "B — EfficientNetB0":  {"macro_auc": 0.96, "mel_recall": 0.85},
                "C — MobileNetV2":     {"macro_auc": 0.94, "mel_recall": 0.82},
            }
    """


    df = pd.DataFrame(results).T.reset_index()
    df.columns = ["Model", "Macro AUC", "Mel recall"]

    colors = ["#7F77DD", "#1D9E75", "#D85A30"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Model comparison — test set")

    axes[0].bar(df["Model"], df["Macro AUC"], color=colors[: len(df)])
    axes[0].set_ylim(0, 1)
    axes[0].set_title("Macro AUC-ROC")
    axes[0].tick_params(axis="x", rotation=15)

    axes[1].bar(df["Model"], df["Mel recall"], color=colors[: len(df)])
    axes[1].axhline(0.8, color="red", linestyle="--", lw=1, label="0.80 target")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Melanoma recall  (clinical priority)")
    axes[1].legend()
    axes[1].tick_params(axis="x", rotation=15)

    plt.tight_layout()
    plt.show()

    print(df.to_string(index=False))

### Data Configuration

In [4]:
# COLAB

train_path = os.path.join(base_path, "augmented_metadata.csv")
val_path = os.path.join(base_path, "val_split.csv")
test_path = os.path.join(base_path, "test_split.csv")

train_df = pd.read_csv(train_path, sep=",")[['image_id', 'dataset', 'lesion_id','image_path', 'dx', 'dx_encoded']]
val_df = pd.read_csv(val_path, sep=",")[['image_id', 'dataset', 'lesion_id','image_path', 'dx_encoded']]
test_df = pd.read_csv(test_path, sep=",")[['image_id', 'dataset', 'lesion_id','image_path', 'dx_encoded']]

val_df = val_df.rename(columns={'cleaned_path': 'image_path'})
test_df = test_df.rename(columns={'cleaned_path': 'image_path'})

train_df['dx_encoded'] = train_df['dx'].map(label2idx).astype(int)

train_df['image_path'] = train_df['image_path'].apply(lambda x: base_path + x[6:].replace('\\', '/'))
val_df['image_path'] = val_df['image_path'].apply(lambda x: base_path + x[6:].replace('\\', '/'))
test_df['image_path'] = test_df['image_path'].apply(lambda x: base_path + x[6:].replace('\\', '/'))

In [ ]:
# Load the CSVs
train_df = pd.read_csv('data/augmented_metadata.csv')[['image_id', 'dataset', 'lesion_id','image_path', 'dx', 'dx_encoded']]
val_df = pd.read_csv('data/val_split.csv')[['image_id', 'dataset', 'lesion_id','cleaned_path', 'dx_encoded']]
test_df = pd.read_csv('data/test_split.csv')[['image_id', 'dataset', 'lesion_id','cleaned_path','dx_encoded']]

val_df = val_df.rename(columns={'cleaned_path': 'image_path'})
test_df = test_df.rename(columns={'cleaned_path': 'image_path'})

train_df['dx_encoded'] = train_df['dx'].map(label2idx).astype(int)

In [5]:
N_CLASSES  = len(np.unique(train_df.dx_encoded))
BATCH_SIZE = 128

# Build datasets
train_ds = make_dataset_new(train_df, shuffle=True, repeat=True, batch_size=BATCH_SIZE)
val_ds   = make_dataset_new(val_df, batch_size=BATCH_SIZE)
test_ds  = make_dataset_new(test_df, batch_size=BATCH_SIZE)

STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE
class_weights_dict = make_class_weights(train_df)

In [6]:
print(f"Steps per epoch: {STEPS_PER_EPOCH}")
print(f"Class weights: {class_weights_dict}")

Steps per epoch: 98
Class weights: {0: np.float64(1.4493569131832797), 1: np.float64(1.2340862422997947), 2: np.float64(0.8370473537604457), 3: np.float64(2.504166666666667), 4: np.float64(0.8727008712487899), 5: np.float64(0.42867332382310985), 6: np.float64(2.3415584415584414)}


### Model Configuration

In [8]:
def build_model():
    model = keras.Sequential([
        keras.layers.Conv2D(32, 3, activation="relu", padding="same", input_shape=(224, 224, 3)),
        keras.layers.MaxPooling2D(2),
        keras.layers.Dropout(0.25),

        keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Dropout(0.25),

        keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Dropout(0.3),

        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(N_CLASSES)  # NO softmax!
    ])
    return model

In [16]:
# 1. Build the custom model
custom_model = build_model()
custom_model.summary()

# 2. Compile (Standard LR for from-scratch training)
custom_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy", BalancedAccuracy(N_CLASSES)]
)

# 3. Callbacks (Give it a bit more patience since it learns from scratch)
my_callbacks = get_callbacks(
    checkpoint_path="checkpoints/model_CUSTOM_best.weights.h5",
    model=custom_model,    
    max_diff=0.15,       
    patience_es=15, 
    patience_lr=5
)

# 4. Train the whole network at once!
history_custom = custom_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=my_callbacks
)

# 5. Evaluate
plot_history(history_custom, "Model Custom CNN — Training")
results_custom = evaluate_model(custom_model, test_ds, label2idx, "Model Custom CNN")

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_9 (Conv2D)           (None, 224, 224, 32)      896       
                                                                 
 max_pooling2d_9 (MaxPooling  (None, 112, 112, 32)     0         
 2D)                                                             
                                                                 
 dropout_12 (Dropout)        (None, 112, 112, 32)      0         
                                                                 
 conv2d_10 (Conv2D)          (None, 112, 112, 64)      18496     
                                                                 
 max_pooling2d_10 (MaxPoolin  (None, 56, 56, 64)       0         
 g2D)                                                            
                                                                 
 dropout_13 (Dropout)        (None, 56, 56, 64)       

KeyboardInterrupt: 